# Track-wise dataset — enriched 0.0625°, 30-minute, 1-minute decisions

This notebook analyzes the completed CPR-gating dataset. The preparation pipeline
has already run; the notebook loads `data/EarthCare_trackwise_decisions.parquet`, which now
contains the validated 40-column enriched schema. It does not re-run extraction.

The dataset supports the future RL problem: use GEOS-5 Nature Run diagnostic
proxies and geometry to decide whether to power the CPR, then score that action
against stored target diagnostics. The cloud and radiation fields are idealized
GEOS-5 proxies, not simulated EarthCARE retrievals.

| | final trackwise dataset |
|---|---|
| collection | `inst30mn_2d_met1_Nx` @ **0.0625°** |
| weather cadence | **30 min** |
| decision cadence | **1 min** |
| rows | **351,639** |
| source columns | 27 |
| enriched columns | **40** |
| decision discontinuities | **49** (1,161 missing decision IDs) |
| labels | CPR swath and nadir targets, retained but never ordinary observations |

The enriched additions are `lwtup_fore`, continuous nadir land/water fractions,
cyclic longitude (`sin_lon`, `cos_lon`), and eight target-validity flags. See
`docs/feature_set.md` and `docs/column_registry.yaml` for the complete feature
tiers and limitations.


> **Completed-dataset mode:** the extraction cells below document the original
configuration and remain useful for provenance, but should not be rerun for this
analysis. The notebook loader reads the validated local
`data/EarthCare_trackwise_decisions.parquet` directly. No weather download or re-extraction is
needed to reproduce the analysis and visuals.

The source parquet is now the enriched 40-column dataset. All joins and checks
in the preparation workflow used the `decision` key, preserving the 49 genuine
discontinuities rather than shifting rows across missing decision IDs.


In [ ]:
# Dependencies are already installed; no package download is needed.

import gc
import os
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- weather window ------------------------------------------------------
# 8 months from the wettest point of the G5NR record. June-July 2005 is the
# wettest consecutive 2-month window (area-weighted global mean prectot
# 3.204 mm/day, highest of all 24 months), so a June start opens on peak
# monsoon/ITCZ and sweeps through NH autumn into winter.
# The G5NR 30-min axis starts 2005-05-15 21:30, hence the step offset.
FIELD_START = 803                      # G5NR step of 2005-06-01 15:00 (same UTC time-of-day as startdate)
N_FIELDS    = 11760                    # 245 days -> 2006-02-01 15:00 (exclusive)
G5_STEP     = timedelta(minutes=30)    # G5NR native cadence
DECISION    = timedelta(minutes=1)     # the CPR on/off decision interval
SUB_STEP    = timedelta(seconds=1)     # 1 s ~= 6.8 km ~= one 0.0625 deg cell,
                                       # which gives a CONTIGUOUS nadir track

# ---- fetch settings ------------------------------------------------------
# Measured against NCCS: ~1.0 s of latency per VARIABLE, and ~0.058 MB/s per
# CONNECTION. So big bounding boxes are ruinous while concurrency is nearly
# free, and many small requests beat one large one. Full 1-minute sampling is
# therefore affordable and no decision sampling is required.
DEC_PER_FIELD = 30      # 30 = EVERY 1-minute decision (no sampling)
GROUP         = 3       # decisions per request; small keeps the bbox on the track
WORKERS       = 18      # PROCESSES. Threads do NOT help (netCDF-C serialises).
                        # ~11x serial throughput and ~2.6 GB RSS on an M4/16GB;
                        # scaling flattens past ~12 (NCCS shapes per source IP),
                        # so on a cluster more NODES beat more local processes.
CACHE         = "cache_8mo_jun2005_aligned"
CHECKPOINT    = 200     # fields per parquet part

G5NR_URL = ("https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/"
            "0.0625_deg/inst/inst30mn_2d_met1_Nx")

# MSI-observable -> may enter the agent's observation vector
MSI = ["tautot", "tauhgh", "taulow", "cldtot", "cldhgh", "cldmid",
       "cldlow", "cldtmp", "lwtup"]
# CPR / ground truth -> label only, NEVER a feature.
# `cape` and `precsno` were dropped: 20-30% fill, never used as features, and
# ~1 s of latency each on every request (~3 h over the run).
LAB = ["prectot", "preccon", "iwp", "lwp"]
VARS = MSI + LAB
assert not (set(MSI) & set(LAB)), "a variable cannot be both feature and label"

startdate = datetime(2026, 10, 15, 15, 0, tzinfo=timezone.utc)  # ground-track clock: first 15:00 UTC after
                                                                # the TLE epoch, same time-of-day as FIELD_START

# decision indices actually sampled: DEC_PER_FIELD consecutive minutes per field
PER_FIELD = int(G5_STEP / DECISION)          # 30 decisions exist per field
DEC_PER_FIELD = min(DEC_PER_FIELD, PER_FIELD)
SAMPLED = np.array([f * PER_FIELD + k
                    for f in range(N_FIELDS) for k in range(DEC_PER_FIELD)])
N_ROWS = len(SAMPLED)
N_REQ = int(np.ceil(N_ROWS / GROUP))

# per request: per-variable latency, plus transfer of a ~0.94 MB slab
SECS_PER_REQ = len(VARS) * 1.0 + 0.94 * len(VARS) / 15 / 0.27
print(f"Jun 2005 - Jan 2006 | {N_FIELDS:,} weather fields x {DEC_PER_FIELD} "
      f"decisions = {N_ROWS:,} rows")
print(f"{len(VARS)} vars ({len(MSI)} features + {len(LAB)} labels)")
print(f"{N_REQ:,} requests x ~{SECS_PER_REQ:.0f}s = {N_REQ*SECS_PER_REQ/3600:.0f} h serial")
# Estimated from MEASURED BLOCK TIMES, not from a burst benchmark: short bursts
# mispredicted this run in BOTH directions (they suggested 194 h).
BLOCK_MIN = 24                               # measured minutes per 200-field block
N_BLOCKS = -(-N_FIELDS // CHECKPOINT)
print(f"    {N_BLOCKS} blocks x ~{BLOCK_MIN} min (measured) = "
      f"{N_BLOCKS*BLOCK_MIN/60:.0f} h ({N_BLOCKS*BLOCK_MIN/1440:.1f} days)")
print(f"    worker RSS ~142 MB -> ~{WORKERS*0.142:.1f} GB total")
print(f"output ~{N_ROWS*25*8/1e6:.0f} MB | peak memory ~1 MB per worker (one slab)")

In [ ]:
from tatc import config, utils
# Footprints (TATC) and nadir points (Skyfield) must share one propagation;
# TATC reuses a detected repeat cycle by default, so pin it off.
config.rc.repeat_cycle_for_orbit_track = False
from tatc.schemas import PointedInstrument, Satellite, TwoLineElements

ROLL = 5.760868        # MSI roll angle, degrees

earthcare = Satellite(
    name="EarthCare",
    orbit=TwoLineElements(tle=[      # CelesTrak GP, epoch 2026-10-14 21:54 UTC
        "1 59908U 24101A   26287.91308245  .00003560  00000+0  50729-4 0  9996",
        "2 59908  97.0045  52.9401 0002233  96.7262 263.4249 15.56977907135196",
    ]),
    instruments=[PointedInstrument(
        name="MSI",
        field_of_regard=utils.swath_width_to_field_of_regard(394e3, 150e3) + 2 * ROLL,
        cross_track_field_of_view=utils.swath_width_to_field_of_view(394e3, 150e3, ROLL),
        along_track_field_of_view=utils.swath_width_to_field_of_view(394e3, 10e3, ROLL),
        roll_angle=ROLL, is_rectangular=True)],
)
satellites = [earthcare]

In [ ]:
import xarray as xr

# HTTP.TIMEOUT turns a stalled DAP transfer into a catchable exception.
dodsrc = os.path.expanduser("~/.dodsrc")
settings = open(dodsrc).read() if os.path.exists(dodsrc) else ""
if "HTTP.TIMEOUT" not in settings:
    with open(dodsrc, "a") as f:
        f.write("HTTP.TIMEOUT=300\n")
    print("added HTTP.TIMEOUT=300 to ~/.dodsrc (restart kernel if DAP already used)")

def open_g5nr(attempts=5):
    """Open the OPeNDAP collection, retrying with a widening pause."""
    for a in range(attempts):
        try:
            return xr.open_dataset(G5NR_URL, decode_times=True)
        except Exception as e:
            print(f"open failed {a+1}/{attempts}: {e}", flush=True)
            time.sleep(15 * (a + 1))
    raise RuntimeError("could not open G5NR")

dataset = open_g5nr()
g5_times = pd.to_datetime(dataset["time"].values)
g5_lat = dataset["lat"].values
g5_lon = dataset["lon"].values
print(f"{G5NR_URL.rsplit('/',1)[-1]}: {len(g5_times):,} steps, "
      f"{g5_times[0]} -> {g5_times[-1]}")
print(f"grid {len(g5_lat)} x {len(g5_lon)}  (dlat {abs(g5_lat[1]-g5_lat[0]):.4f}°)")
missing = [v for v in VARS if v not in dataset.variables]
assert not missing, f"not in this collection: {missing}"
print(f"all {len(VARS)} requested variables present")

In [ ]:
from tatc.analysis import compute_ground_track
from joblib import Parallel, delayed
from skyfield.api import EarthSatellite, wgs84
from tatc.constants import timescale

SUB_PER_DEC = int(DECISION / SUB_STEP)

def part_path(start):
    """Parquet part holding the block of fields that begins at `start`."""
    return f"{CACHE}/rows_{start // CHECKPOINT:05d}.parquet"

# (first_field, last_field + 1) for each block; one block -> one parquet part
BLOCKS = [(b, min(b + CHECKPOINT, N_FIELDS)) for b in range(0, N_FIELDS, CHECKPOINT)]

def pending_blocks():
    """The blocks whose parquet part is not on disk yet."""
    return [(a, b) for a, b in BLOCKS if not os.path.exists(part_path(a))]

# Skip the footprint/nadir precompute when every block is already cached: cell 7
# only consumes `tracks`/`nad_lat`/`nad_lon` for blocks it still has to fetch,
# and for 8 months (352,800 decisions) computing them anyway ran the kernel out
# of memory on a consolidation-only run.
todo = pending_blocks()

if not todo:
    print(f"all {len(BLOCKS)} blocks already cached in {CACHE}/ -> skipping "
          f"footprint/nadir precompute (nothing left to fetch)")
    tracks = pd.DataFrame({"field": pd.Series(dtype="int64")})
    nad_lat = np.empty((0, 0), dtype="float32")
    nad_lon = np.empty((0, 0), dtype="float32")
    ROW_OF = {}
else:
    # swath footprints, ONLY for the sampled decisions
    def one_decision(i):
        t0 = startdate + int(i) * DECISION
        return compute_ground_track(
            satellites[0],
            pd.date_range(t0, t0 + DECISION, freq=SUB_STEP, inclusive="left"),
            crs="spice")

    clock = time.time()
    tracks = pd.concat(Parallel(-1)(delayed(one_decision)(i) for i in SAMPLED),
                       ignore_index=True)
    tracks["decision"] = SAMPLED
    tracks["field"] = SAMPLED // PER_FIELD
    print(f"{len(tracks):,} sampled footprints in {time.time()-clock:.0f}s "
          f"({tracks.geometry.iloc[0].geom_type})")

    # nadir sub-satellite points for those same decisions
    clock = time.time()
    sat = EarthSatellite(satellites[0].orbit.tle[0], satellites[0].orbit.tle[1],
                         "EarthCare", timescale)
    offsets = np.arange(SUB_PER_DEC) * SUB_STEP.total_seconds()
    secs = (SAMPLED[:, None] * DECISION.total_seconds() + offsets[None, :]).ravel()
    ts = timescale.from_datetimes(
        pd.to_datetime(startdate) + pd.to_timedelta(secs, unit="s"))
    subpoint = wgs84.subpoint(sat.at(ts))
    nad_lat = subpoint.latitude.degrees.astype("float32").reshape(len(SAMPLED), SUB_PER_DEC)
    nad_lon = subpoint.longitude.degrees.astype("float32").reshape(len(SAMPLED), SUB_PER_DEC)
    del subpoint, ts, secs
    gc.collect()

    # one sample per second should land about one 0.0625 deg cell further on
    step_km = np.hypot(np.diff(nad_lat[0]) * 111.32,
                       np.diff(nad_lon[0]) * 111.32 * np.cos(np.radians(nad_lat[0][:-1])))
    print(f"nadir: {nad_lat.size:,} points in {time.time()-clock:.0f}s "
          f"({nad_lat.nbytes*2/1e6:.0f} MB) | spacing {step_km.mean():.2f} km "
          f"-> {'contiguous' if step_km.mean() <= 1.5*0.0625*111.32 else 'GAPS'}")

    # decision -> row index in nad_lat/nad_lon
    ROW_OF = {int(d): i for i, d in enumerate(SAMPLED)}

## Streaming extraction

For each group of `GROUP` consecutive decisions:

1. take the union bounding box (consecutive 1-minute segments are nearly collinear, so
   the box hugs the track — this is why `GROUP` is small)
2. fetch that slab for the nearest 30-minute timestep
3. reduce each decision to **one row**: MSI features averaged over the swath, labels taken
   as the peak over the swath *and* separately over the nadir track
4. free the slab

Nothing global is ever materialised, so memory is flat in the length of the run. Parts are
checkpointed to parquet, so a rerun resumes.

In [ ]:
import shapely
from joblib import Parallel, delayed

os.makedirs(CACHE, exist_ok=True)

# Field f maps to G5NR time index FIELD_START + f exactly (one field == one
# 30-min step), so the time mapping is pure index arithmetic. NEVER match
# datetimes: ground-track times run on a 2025 TLE clock while the G5NR axis is
# 2005-2007, so nearest-datetime collapses every request onto the last step
# (that bug once produced a 1-field dataset).
assert FIELD_START + N_FIELDS <= len(g5_times), "window exceeds the G5NR axis"
print(f"time map: field 0..{N_FIELDS-1} -> G5NR steps "
      f"{FIELD_START}..{FIELD_START+N_FIELDS-1} = "
      f"{g5_times[FIELD_START]} .. {g5_times[FIELD_START+N_FIELDS-1]}")

_WDS = None
def worker_dataset(force=False):
    """One DAP handle per worker process, reused; `force` discards a dead one.

    The reset MUST go through this function's own `global _WDS`. Assigning via
    `globals()["_WDS"]` from another function resolves to a different namespace
    under cloudpickle, so the reset silently did nothing: a worker whose handle
    died (laptop sleep) kept handing back the dead handle and every later task
    timed out. Cost: 18 workers, 10 hours, zero rows, on a healthy server.
    """
    global _WDS
    if force and _WDS is not None:
        try:
            _WDS.close()
        except Exception:
            pass
        _WDS = None
    if _WDS is None:
        import xarray as xr
        _WDS = xr.open_dataset(G5NR_URL, decode_times=True)
    return _WDS

def finite(a):
    """Drop NaN/inf and G5NR's large-but-finite fill values.

    `np.isfinite` alone lets through cells that report missing data as a value
    up to float32's max (~3.4e38) -- 640 such rows in the 8-month dataset, e.g.
    a "cloud fraction" of 1.3e35. No variable here is ever physically near 1e6.
    """
    return a[np.isfinite(a) & (np.abs(a) < 1e6)]

def peak(a):
    a = finite(a)
    return float(a.max()) if a.size else float("nan")

def average(a):
    a = finite(a)
    return float(a.mean()) if a.size else float("nan")

def reduce_decision(geom, sub, nlat, nlon):
    """One decision -> one row: MSI meaned over the swath, labels peaked over
    the swath and, separately, over the nadir track."""
    la = sub["lat"].values
    lo = sub["lon"].values
    if la.size == 0 or lo.size == 0:
        return None

    # swath cells: those inside the footprint, or the nearest cell if none are
    LO, LA = np.meshgrid(lo, la)
    inside = shapely.contains_xy(geom, LO, LA)
    if not inside.any():
        c = geom.centroid
        inside = np.zeros_like(LO, dtype=bool)
        inside[np.abs(la - c.y).argmin(), np.abs(lo - c.x).argmin()] = True

    # nadir cells: the nearest grid cell to each sub-satellite point in the slab
    keep = ((nlat >= la.min()) & (nlat <= la.max())
            & (nlon >= lo.min()) & (nlon <= lo.max()))
    if not keep.any():
        keep = np.ones_like(nlat, dtype=bool)
    ni = np.abs(la[:, None] - nlat[keep][None, :]).argmin(axis=0)
    nj = np.abs(lo[:, None] - nlon[keep][None, :]).argmin(axis=0)

    row = {f"{v}_mean": average(sub[v].values[inside]) for v in MSI}
    row["tautot_max"] = peak(sub["tautot"].values[inside])
    for v in LAB:
        arr = sub[v].values
        row[f"{v}_swath"] = peak(arr[inside])
        row[f"{v}_nadir"] = peak(arr[ni, nj])
    row["n_swath_cells"] = int(inside.sum())
    row["n_nadir_cells"] = len(set(zip(ni, nj)))
    return row

def fetch_slab(g5i, geoms, pad=0.13, attempts=4):
    """Load the union bounding box of `geoms` at G5NR step `g5i`, else None."""
    import xarray as xr
    bounds = np.array([g.bounds for g in geoms])      # columns: lon0 lat0 lon1 lat1
    lon0, lat0 = bounds[:, 0].min(), bounds[:, 1].min()
    lon1, lat1 = bounds[:, 2].max(), bounds[:, 3].max()
    lats = slice(lat0 - pad, lat1 + pad)
    for a in range(attempts):
        try:
            field = worker_dataset(force=(a > 0))[VARS].isel(time=g5i)
            if lon1 - lon0 > 180:                     # the segment crossed the seam
                sub = xr.concat(
                    [field.sel(lon=slice(120, 180), lat=lats).load(),
                     field.sel(lon=slice(-180, -120), lat=lats).load()], dim="lon")
            else:
                sub = field.sel(lon=slice(lon0 - pad, lon1 + pad), lat=lats).load()
            # NCCS occasionally answers with a zero-size slab instead of an
            # error; the padded bbox always spans grid cells, so retry it.
            if 0 in (sub.sizes["lat"], sub.sizes["lon"]):
                raise ValueError("empty slab")
            return sub
        except Exception:
            time.sleep(min(4 * 2 ** a, 45))           # retries always get a FRESH handle
    return None

def process_field(fi, geoms, decs, times, nlats, nlons):
    """Up to GROUP consecutive decisions of one 30-min field -> rows. Runs in a
    worker process. `fi` is relative; the G5NR index is FIELD_START + fi.
    Returns None when the download failed (retryable), a list of rows otherwise."""
    g5i = FIELD_START + fi
    sub = fetch_slab(g5i, geoms)
    if sub is None:
        return None
    out = []
    for g, d, t, nla, nlo in zip(geoms, decs, times, nlats, nlons):
        r = reduce_decision(g, sub, nla, nlo)
        if r is None:
            continue
        r.update(decision=int(d), field=int(fi), g5_step=int(g5i),
                 time=t, lat=g.centroid.y, lon=g.centroid.x)
        out.append(r)
    return out

def field_tasks(first, last):
    """Split fields [first, last) into tasks of <= GROUP decisions sharing a step."""
    tasks = []
    for fi in range(first, last):
        seg = by_field.get(fi)
        if seg is None or len(seg) == 0:
            continue
        seg = seg.sort_values("decision")
        for s in range(0, len(seg), GROUP):
            ch = seg.iloc[s:s + GROUP]
            idx = [ROW_OF[int(d)] for d in ch.decision]
            tasks.append((fi, list(ch.geometry), list(ch.decision), list(ch.time),
                          [nad_lat[i] for i in idx], [nad_lon[i] for i in idx]))
    return tasks

by_field = {f: g for f, g in tracks.groupby("field")}
todo = pending_blocks()
print(f"{len(BLOCKS)} blocks | {len(BLOCKS)-len(todo)} cached | {len(todo)} to fetch")
print(f"{N_REQ:,} requests total, {WORKERS} concurrent")

# A task whose in-task retries all fail is re-queued at block level after a
# pause (60 s, 120 s, ... capped at 15 min; ~45 min total), riding out NCCS
# outages and laptop sleeps. A part is written only when every task in the block
# succeeded, so a resume refetches an incomplete block instead of freezing its
# gaps in place. The 2005-06-01 00:00 run had neither and lost 1,161 decisions
# in 49 whole-task gaps.
RETRY_ROUNDS, RETRY_PAUSE = 6, 60
# A task normally takes ~15 s; in-task retries can stretch that to ~10 min.
# Past TASK_TIMEOUT the pool is killed and the whole call counts as failed:
# after a network change (VPN drop) libcurl sits on a dead socket forever and
# ignores HTTP.TIMEOUT, which once froze a block for over an hour.
TASK_TIMEOUT = 900

def run_tasks(tasks):
    """Fetch `tasks` in parallel -> (rows, tasks whose download failed)."""
    try:
        res = Parallel(n_jobs=min(WORKERS, len(tasks)), backend="loky", verbose=0,
                       timeout=TASK_TIMEOUT)(delayed(process_field)(*t) for t in tasks)
    except Exception as e:      # stalled task (TimeoutError) or a killed worker
        print(f"  pool aborted ({type(e).__name__}): {len(tasks)} tasks count as failed")
        res = [None] * len(tasks)
    rows = [r for rs in res if rs for r in rs]
    return rows, [t for t, rs in zip(tasks, res) if rs is None]

clock = time.time()
for done, (a, b) in enumerate(todo, start=1):
    tasks = field_tasks(a, b)
    flat, failed = run_tasks(tasks)
    for rnd in range(1, RETRY_ROUNDS + 1):
        if not failed:
            break
        pause = min(RETRY_PAUSE * 2 ** (rnd - 1), 900)
        print(f"  block {a//CHECKPOINT}: {len(failed)} tasks failed, "
              f"retry round {rnd}/{RETRY_ROUNDS} in {pause} s", flush=True)
        time.sleep(pause)
        rows, failed = run_tasks(failed)
        flat += rows
    want = sum(len(t[2]) for t in tasks)
    if failed:
        print(f"  block {a//CHECKPOINT} INCOMPLETE: {len(failed)} tasks "
              f"({sum(len(t[2]) for t in failed)} decisions) still failing "
              f"-> part NOT saved, rerun to resume", flush=True)
    else:
        if len(flat) != want:
            print(f"  block {a//CHECKPOINT}: {want-len(flat)} decisions had an "
                  f"empty slab (not a download failure)")
        pd.DataFrame(flat).to_parquet(part_path(a), index=False)
    el = time.time() - clock
    print(f"  block {a//CHECKPOINT}: {len(flat):,}/{want:,} rows | {el/60:.0f} min "
          f"| eta {el/done*(len(todo)-done)/3600:.1f} h", flush=True)
    del flat, failed
    gc.collect()
left = pending_blocks()
print(f"\ndone in {(time.time()-clock)/3600:.1f} h | "
      + (f"{len(left)} blocks incomplete -> rerun this cell to resume" if left
         else f"all {len(BLOCKS)} blocks complete"))

In [ ]:
# ---- load the completed enriched dataset ---------------------------------
# Analysis only: this does not concatenate extraction cache parts, contact
# OPeNDAP, or rewrite the source parquet.
DATASET_PATH = "data/EarthCare_trackwise_decisions.parquet"
MM = 3600.0        # kg m-2 s-1 -> mm/hr; every cell below uses this

rows = pd.read_parquet(DATASET_PATH)
assert rows.shape == (351_639, 40), rows.shape
assert rows.decision.is_unique
assert rows.decision.is_monotonic_increasing
assert rows.g5_step.nunique() > 1

# one validity flag per label, per swath/nadir side
validity_flags = [f"{v}_{side}_valid" for v in LAB for side in ("swath", "nadir")]
assert set(validity_flags).issubset(rows.columns)

gaps = (rows.decision.diff().dropna() != 1).sum()
print(f"loaded {len(rows):,} rows x {rows.shape[1]} enriched columns")
print(f"weather fields: {rows.field.nunique():,} | decision discontinuities: {gaps:,}")
print(f"target-invalid entries: {(~rows[validity_flags]).sum().sum():,} | "
      f"decisions with any invalid target: {(~rows[validity_flags].all(axis=1)).sum():,}")
print(f"mixed-surface decisions (0.05 < land_frac_nadir < 0.95): "
      f"{((rows.land_frac_nadir > .05) & (rows.land_frac_nadir < .95)).mean()*100:.2f}%")
rows.head()

In [ ]:
# ---- CLASS BALANCE: valid swath vs nadir targets -------------------------
TARGETS = ["prectot_swath", "prectot_nadir", "preccon_swath", "preccon_nadir"]
THRESHOLDS = [("any > 0", 0.0), (">= 0.1 mm/hr", 0.1), (">= 0.5 mm/hr", 0.5),
              (">= 1 mm/hr", 1.0), (">= 5 mm/hr", 5.0), (">= 10 mm/hr", 10.0)]

print(f"{len(rows):,} decisions ({DECISION.total_seconds()/60:.0f}-minute interval)\n")
print(f"{'threshold':<20}{'prectot swath':>20}{'prectot nadir':>20}"
      f"{'preccon swath':>20}{'preccon nadir':>20}")
for name, mm in THRESHOLDS:
    cells = []
    for c in TARGETS:
        valid = rows[c + "_valid"]
        hit = rows.loc[valid, c] > mm / MM
        cells.append(f"{int(hit.sum()):>7,} ({100*hit.mean():4.1f}%, n={int(valid.sum()):,})")
    print(f"  {name:<18}" + "".join(f"{c:>20}" for c in cells))

print("\nThe nadir column is what the CPR would actually measure; the swath column")
print("credits it with precipitation it never flies over. Validity flags exclude")
print("missing/corrupt targets from every positive-rate calculation.")
for c in ["prectot_swath", "prectot_nadir"]:
    valid = rows[c + "_valid"]
    rate = (rows.loc[valid, c] > 1.0 / MM).mean()
    print(f"  {c:<15} at >=1 mm/hr: {100*rate:.1f}% positive among {int(valid.sum()):,} valid targets")

In [ ]:
# ---- learnability: always-available features vs valid targets ------------
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# The registry's always-available candidates. CPR-conditional diagnostics and
# target columns are deliberately excluded.
FEATS = [
    "cldtot_mean", "cldtmp_mean", "tautot_mean", "tautot_max",
    "lwtup_mean", "lwtup_fore", "solar_hour", "lat", "sin_lon", "cos_lon",
    "land_frac_nadir", "ocean_frac_nadir",
]
RAIN = 1.0 / MM        # the 1 mm/hr label threshold, in kg m-2 s-1

def auc(score, y):
    """ROC-AUC over the finite scores, or NaN if the labels collapse to one class."""
    m = np.isfinite(score)
    return roc_auc_score(y[m], score[m]) if len(set(y[m])) == 2 else float("nan")

def distance_from_chance(scored):
    """Rank key: how far a single-feature AUC sits from 0.5. NaN counts as chance."""
    a = scored[0]
    return abs((a if np.isfinite(a) else 0.5) - 0.5)

for lab in ["prectot_nadir", "prectot_swath"]:
    valid = rows[lab + "_valid"].to_numpy()
    y = (rows.loc[valid, lab] > RAIN).astype(int).to_numpy()
    print(f"\n=== label: {lab} >= 1 mm/hr ({100*y.mean():.1f}% positive; {len(y):,} valid rows) ===")
    scored = [(auc(rows.loc[valid, f].to_numpy(dtype=float), y), f) for f in FEATS]
    for a, f in sorted(scored, key=distance_from_chance, reverse=True)[:8]:
        print(f"  {f:<20}{a:6.3f}" + (f"   (inverted {1-a:.3f})" if a < 0.5 else ""))

valid = rows["prectot_nadir_valid"].to_numpy()
X = rows.loc[valid, FEATS].replace([np.inf, -np.inf], np.nan)
X = ((X - X.mean()) / X.std()).fillna(0).values
y = (rows.loc[valid, "prectot_nadir"] > RAIN).astype(int).values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear").fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]
print(f"\nlogistic on {len(FEATS)} always-available features -> valid nadir prectot >= 1 mm/hr")
print(f"  ROC-AUC {roc_auc_score(yte,p):.3f} | avg-precision "
      f"{average_precision_score(yte,p):.3f} (base rate {yte.mean():.3f})")

In [ ]:
# ---- valid-target distributions and class balance ------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

sw = rows.loc[rows.prectot_swath_valid, "prectot_swath"].clip(lower=1e-9)
na = rows.loc[rows.prectot_nadir_valid, "prectot_nadir"].clip(lower=1e-9)
ax[0].hist(np.log10(sw * MM), bins=60, alpha=.6, label="swath")
ax[0].hist(np.log10(na * MM), bins=60, alpha=.6, label="nadir")
ax[0].axvline(0, color="k", ls="--", lw=1)
ax[0].set_xlabel("log10 peak prectot (mm/hr)"); ax[0].set_ylabel("valid decisions")
ax[0].set_title("Valid target distribution: swath vs nadir"); ax[0].legend()

th = np.array([0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10])
for c, lb in [("prectot_swath", "swath"), ("prectot_nadir", "nadir")]:
    vals = rows.loc[rows[c + "_valid"], c]
    ax[1].plot(th, [100 * (vals > t / MM).mean() for t in th], marker="o", label=lb)
ax[1].axhline(50, color="k", ls="--", lw=1)
ax[1].set_xscale("log"); ax[1].set_xlabel("threshold (mm/hr)")
ax[1].set_ylabel("% positive among valid targets")
ax[1].set_title("Valid-target class balance vs threshold"); ax[1].legend()
plt.tight_layout(); plt.show()

## Visuals and exports\n\nRun these after the loader cell once `cache_junjul2005/` has parts.

In [ ]:
# ---- cartopy visuals (ports of the original notebook's figures) ----------
# Dependencies are already installed; no package download is needed.
import seaborn as sns
import cartopy.crs as ccrs

# Balanced label: fire the CPR when the nadir target beats its own median.
nadir_ok = rows.prectot_nadir_valid
THR = rows.loc[nadir_ok, "prectot_nadir"].quantile(0.50)
print(f"balanced threshold: {THR*MM:.3f} mm/hr "
      f"({100*(rows.prectot_nadir>THR).mean():.1f}% positive)")
rows["fire"] = np.nan
rows.loc[nadir_ok, "fire"] = (rows.loc[nadir_ok, "prectot_nadir"] > THR).astype(int)

# --- 1-2. KDE maps: precipitation, then cloud cover (originals 17-19) ---
for col, title in [
    ("prectot_nadir", "KDE of nadir precipitation diagnostic across the 8-month trackwise dataset"),
    ("tautot_mean",   "KDE of GEOS-5 TAUTOT diagnostic proxy along track (8-month dataset)"),
]:
    d = rows[np.isfinite(rows[col]) & (rows[col] > 0)]
    fig, ax = plt.subplots(figsize=(11, 5), subplot_kw={"projection": ccrs.PlateCarree()})
    try:
        sns.kdeplot(data=d, x="lon", y="lat", weights=col, fill=True,
                    cmap="turbo", bw_adjust=0.25, levels=14, ax=ax)
    except ValueError as e:
        # skewed weights can make the contour levels non-increasing; a weighted
        # hexbin of the same data carries the same information
        print(f"  kdeplot degenerate ({e}); falling back to hexbin for {col!r}")
        ax.hexbin(d["lon"], d["lat"], C=d[col], gridsize=180, cmap="turbo",
                  transform=ccrs.PlateCarree(), reduce_C_function=np.sum)
    ax.coastlines(linewidth=0.6); ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
    ax.set_title(title); ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_xticks(range(-180, 181, 30)); ax.set_yticks(range(-90, 91, 30))
    ax.gridlines(draw_labels=False, linewidth=0.3)
    plt.tight_layout(); plt.show()

# --- 3. decisions along track, coloured by the CPR decision ---
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree()})
sub = rows.iloc[::max(len(rows)//12000, 1)]          # thin for legibility
off = sub[sub.fire == 0]
on = sub[sub.fire == 1]
ax.scatter(off.lon, off.lat, s=1.5, c="#A50026", alpha=.45,
           transform=ccrs.PlateCarree(), label="CPR off")
ax.scatter(on.lon, on.lat, s=1.5, c="#006837", alpha=.65,
           transform=ccrs.PlateCarree(), label="CPR on")
ax.coastlines(linewidth=0.6); ax.set_global()
ax.set_title(f"1-minute CPR decisions along track "
             f"(green = nadir precip > {THR*MM:.2f} mm/hr)")
ax.gridlines(draw_labels=False, linewidth=0.3)
ax.legend(loc="lower left", markerscale=6, framealpha=.9)
plt.tight_layout(); plt.show()

# --- 4. precipitation vs solar hour (originals 15-16) ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h = rows.assign(sh=rows.solar_hour.round()).groupby("sh").agg(
        m=("prectot_nadir", "mean"), p=("fire", "mean")).reset_index()
ax[0].plot(h.sh, h.m * MM, marker=".", color="b")
ax[0].set_xlabel("Solar Hour (0-23)"); ax[0].set_ylabel("mean nadir prectot (mm/hr)")
ax[0].set_title("Precipitation vs solar hour"); ax[0].set_xticks(range(0, 25, 3))
try:
    sns.kdeplot(data=rows, x="solar_hour", y=rows.prectot_nadir*MM, fill=True,
                cmap="Reds", bw_adjust=0.6, ax=ax[1])
except ValueError as e:
    print(f"  kdeplot degenerate ({e}); falling back to hexbin")
    ax[1].hexbin(rows.solar_hour, rows.prectot_nadir*MM, gridsize=60, cmap="Reds")
ax[1].set_xlim(0, 23); ax[1].set_ylim(0, 5); ax[1].set_xticks(range(0, 25, 3))
ax[1].set_xlabel("Solar Hour (0-24)"); ax[1].set_ylabel("nadir prectot (mm/hr)")
ax[1].set_title("Density")
plt.tight_layout(); plt.show()

# --- 5. latitude bands (original 24) ---
band = pd.cut(rows.lat, [-90,-75,-45,-15,15,45,75,90],
              labels=["<-75","-75:-45","-45:-15","tropics","15:45","45:75",">75"],
              include_lowest=True)
by_band = rows.groupby(band, observed=True)
rep = pd.DataFrame({"decisions": by_band.size(),
                    "CPR-on %": 100 * by_band.fire.mean(),
                    "mean mm/hr": MM * by_band.prectot_nadir.mean()})
display(rep.round(2))
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(rows.loc[rows.fire==1, "lat"], bins=72, color="#006837", alpha=.8)
ax.set_xlabel("latitude"); ax.set_ylabel("CPR-on decisions")
ax.set_title("Where the agent should fire the CPR")
plt.tight_layout(); plt.show()

## Enriched-dataset diagnostics

These visuals are derived from the final 40-column parquet. They make the target
validity mask, static land/water context, cyclic longitude representation, and
verified one-minute sequential structure visible before any RL environment is
built.


In [ ]:
# ---- enriched diagnostics: validity, surface, longitude, autocorrelation --

# 1. Missing/corrupt target validity counts. False means missing or a finite
#    abs(value) >= 1e6 sentinel; no target is imputed.
invalid_counts = (~rows[validity_flags]).sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
invalid_counts.plot.bar(ax=ax, color="#b2182b")
ax.set_ylabel("invalid entries"); ax.set_title("Target-validity audit")
ax.tick_params(axis="x", rotation=45); plt.tight_layout(); plt.show()

# 2. Continuous surface fractions. Mixed-surface decisions are defined as
#    0.05 < land_frac_nadir < 0.95; they average 60 one-second samples and are
#    not necessarily individual shoreline grid cells.
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rows.land_frac_nadir, bins=40, alpha=.7, label="land fraction", color="#8c510a")
ax.hist(rows.ocean_frac_nadir, bins=40, alpha=.6, label="water fraction", color="#01665e")
ax.set_xlabel("continuous fraction"); ax.set_ylabel("decisions")
ax.set_title("Nadir land/water fractions"); ax.legend(); plt.tight_layout(); plt.show()

# 3. Longitude seam check: raw longitude is retained, while sin/cos encode it
#    continuously around +/-180 degrees.
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(rows.sin_lon.iloc[::20], rows.cos_lon.iloc[::20], s=.5, alpha=.2)
t = np.linspace(0, 2*np.pi, 400); ax.plot(np.sin(t), np.cos(t), "k--", lw=.8)
ax.set_aspect("equal"); ax.set_xlabel("sin_lon"); ax.set_ylabel("cos_lon")
ax.set_title("Cyclic longitude representation"); plt.tight_layout(); plt.show()

# 4. Verified autocorrelation diagnostic. Pairs crossing decision gaps were
#    excluded before calculating these values.
acf = {
    "preccon_nadir": [0.116024, 0.056602, 0.038470, 0.024907, -0.003190, -0.016144],
    "prectot_nadir": [0.171179, 0.060293, 0.015995, 0.011164, 0.005938, -0.007796],
    "tautot_mean": [0.628451, 0.370135, 0.229986, 0.066772, -0.054396, -0.030714],
}
lags = [1, 2, 3, 5, 10, 30]
fig, ax = plt.subplots(figsize=(8, 4))
for name, values in acf.items():
    ax.plot(lags, values, marker="o", label=name)
ax.axhline(np.exp(-1), color="k", ls="--", lw=.8, label="1/e")
ax.axhline(0, color="0.6", lw=.7); ax.set_xscale("log")
ax.set_xticks(lags); ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("lag (one-minute decisions)"); ax.set_ylabel("autocorrelation")
ax.set_title("Sequential structure at the saved cadence"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ---- exports -------------------------------------------------------------
# CZML is the deliverable here: Cesium needs TIME-DYNAMIC data to animate an
# orbit. (GeoJSON is static and cannot express "the satellite is here at t" --
# that is why it is not produced.) CSV is kept for kepler.gl / analysis.
import json

ALT = 394_000.0            # EarthCARE mean altitude, metres
CZML_DAYS = 3              # animate this many days; ~0.25 MB/day
BEAM_EVERY = 1             # draw the CPR beam on every Nth shown decision
SHOW_MM_HR = 1.0           # CZML colour/beam threshold (display only)

# The visuals cell already built `fire`; rebuild it only if that cell was skipped.
if "fire" not in rows.columns:
    nadir_ok = rows.prectot_nadir_valid
    THR = rows.loc[nadir_ok, "prectot_nadir"].quantile(0.50)
    rows["fire"] = np.nan
    rows.loc[nadir_ok, "fire"] = (rows.loc[nadir_ok, "prectot_nadir"] > THR).astype(int)
# report the threshold as the smallest firing value actually exported
THR = rows.loc[rows.fire == 1, "prectot_nadir"].min()

# --- CSV (kepler.gl auto-detects lat/lon; far lighter than any geo format) ---
rows.to_csv("trackwise_decisions_full.csv", index=False)
rows.iloc[::max(len(rows)//20000, 1)].to_csv("trackwise_decisions_kepler.csv", index=False)

# --- CZML ---------------------------------------------------------------
# `fire` (median split) stays the balanced training label; coloured by it, ~50%
# of the globe is green. The CZML uses its own physical threshold instead.
# Not added to `rows`, so the CSVs above are unchanged.
show = pd.Series(np.nan, index=rows.index)
ok = rows.prectot_nadir_valid
show[ok] = (rows.loc[ok, "prectot_nadir"] * MM >= SHOW_MM_HR).astype(int)

def iso(t):
    return t.strftime("%Y-%m-%dT%H:%M:%SZ")

t0 = rows.time.min()
seg = (rows.assign(show=show)
          [rows.time < t0 + pd.Timedelta(days=CZML_DAYS)].sort_values("time"))
epoch = iso(t0)
stop = iso(seg.time.max())

# satellite position samples: [seconds_since_epoch, lon, lat, alt, ...]
samples = []
for r in seg.itertuples():
    samples += [float((r.time - t0).total_seconds()),
                round(float(r.lon), 4), round(float(r.lat), 4), ALT]

czml = [
    {"id": "document", "name": "EarthCARE trackwise CPR gating (July 2025)", "version": "1.0",
     "clock": {"interval": f"{epoch}/{stop}", "currentTime": epoch,
               "multiplier": 300, "range": "LOOP_STOP",
               "step": "SYSTEM_CLOCK_MULTIPLIER"}},
    {"id": "earthcare", "name": "EarthCARE",
     "description": "MSI images continuously; the agent decides each minute "
                    "whether to power the CPR.",
     "availability": f"{epoch}/{stop}",
     "position": {"epoch": epoch, "cartographicDegrees": samples,
                  "interpolationAlgorithm": "LAGRANGE", "interpolationDegree": 5},
     "point": {"color": {"rgba": [255, 255, 255, 255]}, "pixelSize": 12,
               "outlineColor": {"rgba": [40, 40, 40, 255]}, "outlineWidth": 2},
     "label": {"text": "EarthCARE", "fillColor": {"rgba": [255, 255, 255, 220]},
               "font": "12pt sans-serif", "pixelOffset": {"cartesian2": [14, 0]},
               "horizontalOrigin": "LEFT", "scaleByDistance":
                   {"nearFarScalar": [1.5e6, 1.0, 1.5e7, 0.4]}},
     "path": {"material": {"polylineOutline": {
                  "color": {"rgba": [120, 190, 255, 200]},
                  "outlineColor": {"rgba": [0, 0, 0, 90]}, "outlineWidth": 1}},
              "width": 3, "leadTime": 0, "trailTime": 5400, "resolution": 60}},
]

# ground decisions, plus a nadir "beam" whenever nadir precip >= SHOW_MM_HR
nfire = 0
for r in seg.itertuples():
    s, e = iso(r.time), iso(r.time + pd.Timedelta(minutes=25))
    on = r.show == 1                       # NaN (invalid nadir) is not "on"
    state, color = (("n/a", [150, 150, 150, 110]) if pd.isna(r.show) else
                    ("ON", [0, 190, 80, 200]) if on else
                    ("off", [190, 40, 50, 200]))
    label = "n/a" if pd.isna(r.fire) else int(r.fire)
    czml.append({
        "id": f"d{int(r.decision)}", "availability": f"{s}/{e}",
        "position": {"cartographicDegrees": [float(r.lon), float(r.lat), 0.0]},
        "point": {"pixelSize": 7, "heightReference": "CLAMP_TO_GROUND",
                  "color": {"rgba": color}},
        "description": (f"<b>{iso(r.time)}</b><br>"
                        f"CPR (precip >= {SHOW_MM_HR:g} mm/hr): {state}<br>"
                        f"training label (fire): {label}<br>"
                        f"nadir precip: {r.prectot_nadir*MM:.2f} mm/hr<br>"
                        f"tautot: {r.tautot_mean:.1f} | cloud: {r.cldtot_mean:.2f}")})
    if on:
        nfire += 1
        if nfire % BEAM_EVERY == 0:
            czml.append({
                "id": f"beam{int(r.decision)}",
                "availability": f"{s}/{iso(r.time + pd.Timedelta(minutes=2))}",
                "polyline": {"positions": {"cartographicDegrees":
                                 [float(r.lon), float(r.lat), ALT,
                                  float(r.lon), float(r.lat), 0.0]},
                             "material": {"solidColor": {"color":
                                 {"rgba": [0, 255, 140, 170]}}},
                             "width": 2, "arcType": "NONE"}})

with open("trackwise_decisions.czml", "w") as f:
    json.dump(czml, f)

for fn in ["trackwise_decisions_full.csv", "trackwise_decisions_kepler.csv", "trackwise_decisions.czml"]:
    print(f"  {fn:<28}{os.path.getsize(fn)/1e6:7.2f} MB")
print(f"\nCZML: {CZML_DAYS} days, {len(seg):,} decisions, "
      f"{nfire:,} shown >= {SHOW_MM_HR:g} mm/hr, {len(czml):,} packets")
print(f"training-label threshold {THR*MM:.3f} mm/hr "
      f"({100*seg.fire.mean():.1f}% of these decisions) | clock 300x real time "
      f"-> {CZML_DAYS*24*3600/300/60:.0f} min of playback")
print("\nView: https://sandcastle.cesium.com -> paste a viewer + "
      "Cesium.CzmlDataSource.load('trackwise_decisions.czml'), or drag onto cesium.com/ion")
print("kepler.gl: drag trackwise_decisions_kepler.csv onto https://kepler.gl/demo")